# 01 - Database Population

## 1. Notebook Goal
Create the Phase 2 retrieval database by ingesting train/validation samples into paired Chroma image and text collections.

## 2. Experimental Design
- Build structured multimodal records from CSV annotations and image files.
- Generate embeddings with configured image/text encoders.
- Populate persistent collections in batches.
- Run post-population sanity checks on counts and class statistics.

## 3. Inputs
- dataset/CVPR_2024_dataset_Train/
- dataset/CVPR_2024_dataset_Val/
- dataset_text/train.csv
- dataset_text/val.csv

## 4. Outputs
- chroma_db/ image and text collections used by all Phase 2 experiments.
- Printed diagnostics for record counts and distribution checks.

## 5. Execution Guide
Run cells from top to bottom once. Re-run population cells only when source data or embedding checkpoints change.

### Cell 2 - Environment and Configuration
Purpose: import dependencies, register project paths, load Phase 2 configuration, and define reusable constants needed for data ingestion and database creation.

In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)

### Cell 3 - Build Training and Validation Records
Purpose: read CSV annotations, resolve file paths, and assemble structured multimodal records while reporting missing files before database insertion.

In [ ]:
CONFIG = get_phase2_config()

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"

if not TRAIN_DIR.exists() or not TRAIN_CSV.exists():
    raise FileNotFoundError(
        "Training dataset missing. Expected dataset and dataset_text paths in repo root."
    )

records, missing_examples, total_rows = build_records_from_csv(
    csv_path=TRAIN_CSV,
    split_dir=TRAIN_DIR,
    text_column="text",
    label_column="label",
    text_key="filename",
)

if missing_examples:
    print("Skipped rows with missing image files (up to 10 shown):")
    for item in missing_examples:
        print(f"  - {item}")

print(f"Training records available for DB population: {len(records)} / {total_rows}")

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training records available for DB population: 11685 / 11685


### Cell 4 - Connect to ChromaDB
Purpose: initialize the persistent Chroma client and obtain the target image/text collections that will store embeddings and metadata.

In [ ]:
try:
    client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

expected_count = len(records)
if (
    image_collection.count() == expected_count
    and text_collection.count() == expected_count
):
    print("Collections already populated. Skipping insertion.")
    SKIP_POPULATION = True
else:
    SKIP_POPULATION = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 663.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Cell 5 - Prepare Embedder Models
Purpose: load or configure image and text embedders/checkpoints so each record can be transformed into retrieval-ready vectors.

In [ ]:
if not SKIP_POPULATION:
    batch_size = CONFIG["batch_size"]
    for start in tqdm(range(0, len(records), batch_size), desc="Populating ChromaDB"):
        batch = records[start : start + batch_size]
        ids_img, ids_txt = [], []
        images, docs = [], []
        metadatas = []

        for idx, record in enumerate(batch, start=start):
            image = Image.open(record["image_path"]).convert("RGB")
            image_np = np.array(image, dtype=np.uint8)
            ids_img.append(f"img_{idx}")
            ids_txt.append(f"txt_{idx}")
            images.append(image_np)
            docs.append(record["filename"])
            metadatas.append({"label": record["label"], "filename": record["filename"]})

        try:
            image_collection.add(ids=ids_img, images=images, metadatas=metadatas)
            text_collection.add(ids=ids_txt, documents=docs, metadatas=metadatas)
        except Exception as exc:
            raise RuntimeError(
                f"ChromaDB insert failed for batch starting at {start}: {exc}"
            ) from exc

print("Population step complete.")

Populating ChromaDB: 100%|██████████| 117/117 [24:53<00:00, 12.77s/it]

Population step complete.


### Cell 6 - Populate Image/Text Collections
Purpose: batch-embed records and write vectors plus metadata into the paired Chroma collections for the full train+validation corpus.

In [4]:
print(f"Image collection count: {image_collection.count()}")
print(f"Text collection count: {text_collection.count()}")

Image collection count: 11685
Text collection count: 11685


### Cell 8 - Validate Embedding Statistics
Purpose: compute within-class embedding distance diagnostics to sanity-check representation quality after population.

In [7]:
# Compute exact pairwise cosine-distance stats for all samples in each class (no truncation).
import numpy as np

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"

try:
    client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

BLOCK_SIZE = 512

def pairwise_distance_stats_all(embeddings: np.ndarray, block_size: int = 512):
    emb = np.asarray(embeddings, dtype=np.float64)
    n = emb.shape[0]
    if n < 2:
        return None

    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    emb = emb / norms

    count = 0
    sum_dist = 0.0
    sum_sq_dist = 0.0
    min_dist = np.inf
    max_dist = -np.inf

    for i in range(0, n, block_size):
        left = emb[i : i + block_size]
        for j in range(i, n, block_size):
            right = emb[j : j + block_size]
            sim_block = left @ right.T
            np.clip(sim_block, -1.0, 1.0, out=sim_block)
            dist_block = 1.0 - sim_block
            np.clip(dist_block, 0.0, 2.0, out=dist_block)

            if i == j:
                tri = np.triu_indices(dist_block.shape[0], k=1)
                values = dist_block[tri]
            else:
                values = dist_block.ravel()

            if values.size == 0:
                continue

            count += values.size
            sum_dist += values.sum()
            sum_sq_dist += np.square(values).sum()

            local_min = float(values.min())
            local_max = float(values.max())
            if local_min < min_dist:
                min_dist = local_min
            if local_max > max_dist:
                max_dist = local_max

    mean = sum_dist / count
    variance = max((sum_sq_dist / count) - (mean**2), 0.0)
    std = variance**0.5

    return {
        "pairs": int(count),
        "mean": float(mean),
        "std": float(std),
        "min": float(min_dist),
        "max": float(max_dist),
    }

for class_name in ["Black", "Blue", "Green", "TTR"]:
    results = image_collection.get(where={"label": class_name}, include=["embeddings"])
    embeddings = np.asarray(results["embeddings"], dtype=np.float64)

    stats = pairwise_distance_stats_all(embeddings, block_size=BLOCK_SIZE)
    if stats is None:
        print(f"{class_name}: insufficient samples for pairwise stats (n={len(embeddings)})")
        continue

    print(
        f"{class_name}: n={len(embeddings)}, "
        f"pairs={stats['pairs']}, "
        f"mean={stats['mean']:.3f}, "
        f"std={stats['std']:.3f}, "
        f"min={stats['min']:.3f}, "
        f"max={stats['max']:.3f}"
    )

Black: n=2382, pairs=2835771, mean=0.548, std=0.119, min=0.000, max=0.990
Blue: n=4856, pairs=11787940, mean=0.534, std=0.109, min=0.000, max=1.003
Green: n=2331, pairs=2715615, mean=0.467, std=0.102, min=0.000, max=0.965
TTR: n=2116, pairs=2237670, mean=0.615, std=0.126, min=0.000, max=1.054


### Cell 9 - Optional Scratch/Debug Cell
Purpose: reserved for ad-hoc checks, quick sanity queries, or temporary debugging without modifying the main ingestion pipeline.